In [ ]:
CHUNK_MIN_LEN = 80
CHUNK_MAX_LEN = 400
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
OUTPUT_DIR = "handbook-1.0"
SIMILARITY_THRESHOLD = 0.65
PAGERANK_DAMPING = 0.85

In [18]:
import fitz, re

def load_pdf(path):
    doc = fitz.open(path)
    pages = []
    for i, page in enumerate(doc):
        text = re.sub(r'\s+', ' ', page.get_text())
        pages.append({"page": i+1, "text": text})
    return pages

def chunk_pages(pages):
    chunks = []
    for p in pages:
        splits = re.split(r'(\(\d+\))', p["text"])
        current = ""
        for s in splits:
            if re.match(r'\(\d+\)', s):
                if len(current) > CHUNK_MIN_LEN:
                    chunks.append({"page": p["page"], "text": current.strip()})
                current = s
            else:
                current += " " + s
        if len(current) > CHUNK_MIN_LEN:
            chunks.append({"page": p["page"], "text": current.strip()})
    return chunks

pages = load_pdf("./data/handbook.pdf")
chunks = chunk_pages(pages)

In [19]:
def simhash(text):
    return hash(text)

seen = set()
filtered = []
for c in chunks:
    h = simhash(c["text"])
    if h not in seen:
        seen.add(h)
        filtered.append(c)

chunks = filtered

In [20]:
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer(EMBED_MODEL_NAME)

texts = [c["text"] for c in chunks]
embeddings = embed_model.encode(texts, normalize_embeddings=True)

for i, c in enumerate(chunks):
    c["embedding"] = embeddings[i]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8448.82it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
import networkx as nx

G = nx.Graph()

for i in range(len(chunks)):
    G.add_node(i)

for i in range(len(chunks)):
    for j in range(i+1, len(chunks)):
        sim = float(np.dot(embeddings[i], embeddings[j]))
        if sim > SIMILARITY_THRESHOLD:
            G.add_edge(i, j, weight=sim)

pr = nx.pagerank(G, alpha=PAGERANK_DAMPING)

for i, c in enumerate(chunks):
    c["pagerank"] = pr.get(i, 0.0)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=20000)
tfidf_matrix = vectorizer.fit_transform(texts)

In [ ]:
import os, pickle, json

os.makedirs(OUTPUT_DIR, exist_ok=True)

with open(f"{OUTPUT_DIR}/chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

with open(f"{OUTPUT_DIR}/tfidf.pkl", "wb") as f:
    pickle.dump((vectorizer, tfidf_matrix), f)

embed_model.save(f"{OUTPUT_DIR}/embed_model")